In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 네이버 영화 리뷰 데이터셋 로드
df = pd.read_csv('ratings.txt', sep='\t')

# 결측치 전처리
df = df.dropna()

# 테스트 셋 만들기
train_tmp_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

# 검증 셋 만들기
train_data, val_data = train_test_split(train_tmp_data, test_size=0.2, random_state=42)

print(f'훈련 세트 : {len(train_data)}')
print(f'검증 세트 : {len(val_data)}')
print(f'테스트 세트 : {len(test_data)}')

훈련 세트 : 127994
검증 세트 : 31999
테스트 세트 : 39999


In [11]:
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# 훈련 데이터 기반으로 간단한 공백 기준의 단어 사전(Vocabulary) 구축
vocab_size = 20000
all_words = []
for text in train_data['document']:
    all_words.extend(text.split())

# 빈도수가 높은 순서대로 단어 정렬 후 고유 인덱스 부여 (0은 패딩, 1은 모르는 단어)
counts = Counter(all_words)
vocab = {word: i+2 for i, (word, _) in enumerate(counts.most_common(vocab_size - 2))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

In [12]:
vocab

{'영화': 2,
 '너무': 3,
 '정말': 4,
 '진짜': 5,
 '이': 6,
 '영화.': 7,
 '왜': 8,
 '더': 9,
 '그냥': 10,
 '이런': 11,
 '수': 12,
 '영화를': 13,
 '잘': 14,
 '다': 15,
 '보고': 16,
 '그': 17,
 '영화가': 18,
 '영화는': 19,
 '좀': 20,
 '본': 21,
 '최고의': 22,
 '내가': 23,
 'ㅋㅋ': 24,
 '없는': 25,
 '이건': 26,
 '이렇게': 27,
 '봤는데': 28,
 '이거': 29,
 '평점': 30,
 '완전': 31,
 '있는': 32,
 '이게': 33,
 '좋은': 34,
 '내': 35,
 '다시': 36,
 '보는': 37,
 '평점이': 38,
 '그리고': 39,
 '참': 40,
 '역시': 41,
 '많이': 42,
 '난': 43,
 '한': 44,
 '연기': 45,
 '재밌게': 46,
 '하는': 47,
 '것': 48,
 '쓰레기': 49,
 '또': 50,
 '보면': 51,
 '드라마': 52,
 '아': 53,
 '꼭': 54,
 '가장': 55,
 'ㅋㅋㅋ': 56,
 '마지막': 57,
 '스토리': 58,
 '무슨': 59,
 'ㅠㅠ': 60,
 '같은': 61,
 '없고': 62,
 '만든': 63,
 '없다.': 64,
 '끝까지': 65,
 '볼': 66,
 '안': 67,
 'ㅋ': 68,
 '.': 69,
 '대한': 70,
 '솔직히': 71,
 '넘': 72,
 '하지만': 73,
 '봐도': 74,
 '전혀': 75,
 '말이': 76,
 '뭐': 77,
 '10점': 78,
 '내내': 79,
 '영화의': 80,
 '다른': 81,
 '지금': 82,
 '아주': 83,
 '할': 84,
 '최악의': 85,
 '그래도': 86,
 '뭔가': 87,
 'ㅎㅎ': 88,
 '많은': 89,
 '재미있게': 90,
 '어떻게': 91,
 '..': 92,
 '연기가